# Analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("code"))
import pipeline

os.environ.setdefault("FS_LICENSE", os.path.expanduser("~/tools/license.txt"))
pipeline.require_tools()
print("All required tools found on PATH -- make sure this kernel was started from the "
      "`longi` conda env (conda activate longi && jupyter lab) so dcm2niix/ANTs resolve, "
      "and that FreeSurfer's SetUpFreeSurfer.sh has been sourced so mri_synthstrip/"
      "mri_synthseg resolve too.")

In [ ]:
import os
import shutil
from tqdm import tqdm


In [ ]:
!ls ${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed/30001

# Move Raw Patient together

In [ ]:
# Define paths
raw_root = os.path.join(os.environ["RAW_DATA_DIR"], "Datasets/medical/Brain/OASIS/raw")  # unzip path
clean_root = os.path.join(os.environ["RAW_DATA_DIR"], "Datasets/medical/Brain/OASIS/processed")


# Loop through subfolders: train, val, test
for split in ["train", "valid", "test"]:
    split_path = os.path.join(raw_root, split)
    
    if not os.path.exists(split_path):
        continue  # Skip if subfolder doesn't exist

    # Loop through patient folders like 30960_2110_3518
    for folder in tqdm(os.listdir(split_path)):
        folder_path = os.path.join(split_path, folder)
        if not os.path.isdir(folder_path):
            continue  # Skip if subfolder doesn't exist

        # Extract patient ID from folder name
        patient_id = folder.split("_")[0]
        dest_folder = os.path.join(clean_root, patient_id)
        os.makedirs(dest_folder, exist_ok=True)

        # Move .nii.gz files into clean/{patient_id}/
        for file in os.listdir(folder_path):
            time_point = file.split("/")[-1].split("_")[0]
            dest_folder_t = os.path.join(dest_folder, time_point)
            os.makedirs(dest_folder_t, exist_ok=True)
                        
            if file.endswith(".nii.gz"):
                src_file = os.path.join(folder_path, file)
                dest_file = os.path.join(dest_folder_t, file)

                # Only copy if the file doesn't already exist at the destination
                if not os.path.exists(dest_file):
                    shutil.copy2(src_file, dest_file)  # Use move() to remove original
#                     print("dest_file=", dest_file)
                    
                    
print("✅ All files moved from train/val/test to clean/ folders by patient.")


In [ ]:
!ls $clean_root/*/*/*


In [ ]:
# Download 3Yr Data, Metadata and csv from https://ida.loni.usc.edu/pages/access/search.jsp?tab=collection&project=ADNI&page=DOWNLOADS&subPage=IMAGE_COLLECTIONS
from glob import glob
import os

root = clean_root


all_files = glob(f"{root}/*/*")     #  patients
print("all_files", all_files[0])
len(all_files)

In [ ]:
!ls ${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed/30863

# Install and guidance

In [ ]:
# FreeSurfer setup (superseded -- the hcc conda channel only has FreeSurfer 5.3/6.0,
# which predate mri_synthstrip (7.2+) and mri_synthseg (7.3+). Install FreeSurfer from the
# official .deb instead:
#   https://surfer.nmr.mgh.harvard.edu/pub/dist/freesurfer/<version>/freesurfer_ubuntu24-<version>_amd64.deb
#   sudo apt install ./freesurfer_ubuntu24-<version>_amd64.deb
#   source /usr/local/freesurfer/<version>/SetUpFreeSurfer.sh
#   export FS_LICENSE=~/tools/license.txt
# ANTs and dcm2niix install cleanly from conda-forge into the `longi` env:
#   conda create -n longi -c conda-forge python=3.11 ants dcm2niix nibabel scipy numpy tqdm

In [ ]:
# The expat/itk patch below was a workaround for an old, broken conda-forge ants build.
# ants 2.6.5 from conda-forge (see previous cell) installs and runs cleanly as of this
# writing -- this workaround is no longer needed and is kept only for reference:
#
# !mamba uninstall expat -y
# !mamba install -c conda-forge expat=2.1.0 -y
# !mamba install -c conda-forge itk -y

In [ ]:
# Old tarball-based FreeSurfer 7/8-beta install instructions (superseded -- FreeSurfer now
# ships as a .deb, see the "FreeSurfer setup" cell above). Kept for reference:
#
# sudo apt-get install libpng-dev
# sudo apt install gettext xterm csh tcsh xorg-dev libncurses5 libffi6 libjpeg62
# sudo apt-get install libncurses5 libjpeg62 libtinfo5
#
# Test installation with:
#   freeview -v
#   recon-all -s bert -all

In [ ]:
# intensity-normalization is on PyPI now (no need to build from git):
#   pip install intensity-normalization
# The CLI changed from the old per-method `ws-normalize` script to a single unified command:
#   intensity-normalize whitestripe <input> -o <output> --modality t1
# (pipeline.normalize_intensity, used below, already calls the current CLI.)

# Process


In [ ]:
root       = os.path.join(os.environ["RAW_DATA_DIR"], "Datasets/medical/Brain/OASIS/processed")
output_dir = os.path.join(os.environ["RAW_DATA_DIR"], "Datasets/medical/Brain/OASIS/processed")
reference_brain = os.environ.get("REFERENCE_BRAIN", os.path.expanduser("~/tools/MNI152_T1_1mm_brain.nii.gz"))
assert os.path.exists(reference_brain), f"reference brain not found: {reference_brain}"
# Structure
# root/<subject_id>/<preprocessing>/<date>/<acquisition_id>/<file_name>.nii

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt

# Load the NIfTI image
img = nib.load(reference_brain)
data = img.get_fdata()

# Determine the middle slices for each axis
sagittal_slice = data.shape[0] // 2  # Middle sagittal slice
coronal_slice = data.shape[1] // 2   # Middle coronal slice
axial_slice = data.shape[2] // 2     # Middle axial slice

# Plot the slices
fig, axes = plt.subplots(1, 3, figsize=(6, 2))

# Sagittal view
axes[0].imshow(data[sagittal_slice, :, :], cmap="gray", origin="lower")
axes[0].set_title("Sagittal View")
axes[0].axis("off")

# Coronal view
axes[1].imshow(data[:, coronal_slice, :], cmap="gray", origin="lower")
axes[1].set_title("Coronal View")
axes[1].axis("off")

# Axial view
axes[2].imshow(data[:, :, axial_slice], cmap="gray", origin="lower")
axes[2].set_title("Axial View")
axes[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
sample_file = "${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed/30003/4954/4954_anat2.nii.gz"

import nibabel as nib
import matplotlib.pyplot as plt

# Load the NIfTI image
img = nib.load(sample_file)
data = img.get_fdata()

# Determine the middle slices for each axis
sagittal_slice = data.shape[0] // 2  # Middle sagittal slice
coronal_slice = data.shape[1] // 2   # Middle coronal slice
axial_slice = data.shape[2] // 2     # Middle axial slice

# Plot the slices
fig, axes = plt.subplots(1, 3, figsize=(6, 2))

# Sagittal view
axes[0].imshow(data[sagittal_slice, :, :], cmap="gray", origin="lower")
axes[0].set_title("Sagittal View")
axes[0].axis("off")

# Coronal view
axes[1].imshow(data[:, coronal_slice, :], cmap="gray", origin="lower")
axes[1].set_title("Coronal View")
axes[1].axis("off")

# Axial view
axes[2].imshow(data[:, :, axial_slice], cmap="gray", origin="lower")
axes[2].set_title("Axial View")
axes[2].axis("off")

plt.tight_layout()
plt.show()




In [ ]:
from tqdm import tqdm
from glob import glob
import os

# subjects  = [f for f in glob(os.path.join(root, "**", "*"), recursive=True) if not f.endswith(".xml")]
subjects  = glob(os.path.join(root, "*"))  #[f for f in glob(os.path.join(root, "*"), recursive=True)]

print(subjects)
print("number of subjects:", len(subjects))

In [ ]:
# Step 1: run the preprocessing pipeline in parallel, reusing the same step functions as
# the ADNI script (code/pipeline.py) instead of re-implementing them here.
# CPU-only steps (N4, registration) parallelize across `max_workers` threads; GPU steps
# (SynthStrip, SynthSeg) are serialized through gpu_lock -- a single 11GB card can't
# usefully run two of those at once. Each step raises pipeline.StepError on failure
# (checked subprocess return code) instead of silently leaving a missing/empty output for
# the next step to choke on, and skips itself if its output already exists, so a
# crashed/interrupted run can just be re-invoked.
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import threading

print("Running the preprocessing pipeline in parallel...")

gpu_lock = threading.Lock()
RESOLUTIONS = [1.0, 1.5]  # keep both -- disk is not the constraint here


def _strip_nii_suffix(path: str) -> str:
    for suffix in (".nii.gz", ".nii"):
        if path.endswith(suffix):
            return path[: -len(suffix)]
    return path


def process_nii(nii, root, output_dir):
    """Run the full pipeline on a single raw NIfTI file."""
    if "_step" in nii or "_final" in nii or "_synthseg" in nii:
        return None

    base = _strip_nii_suffix(nii.replace(root, output_dir))
    paths = {
        "biasfield": Path(base + "_step1_biasfield.nii.gz"),
        "stripped": Path(base + "_step2_stripped.nii.gz"),
        "registered": Path(base + "_step3_registered.nii.gz"),
        "synthseg": Path(base + "_synthseg.nii.gz"),
        "final": Path(base + "_final.nii.gz"),
    }

    try:
        pipeline.n4_bias_correction(Path(nii), paths["biasfield"])

        with gpu_lock:
            pipeline.skull_strip(paths["biasfield"], paths["stripped"], gpu=True)

        pipeline.register_to_reference(paths["stripped"], paths["registered"], Path(reference_brain))

        with gpu_lock:
            pipeline.segment(paths["registered"], paths["synthseg"], gpu=True)

        pipeline.normalize_intensity(paths["registered"], paths["final"])

        for res in RESOLUTIONS:
            tag = f"{res:g}mm"
            pipeline.resample(paths["synthseg"], Path(base + f"_synthseg_{tag}.nii.gz"), res, is_segmentation=True)
            pipeline.resample(paths["final"], Path(base + f"_final_{tag}.nii.gz"), res, is_segmentation=False)

        return f"ok: {nii}"
    except pipeline.StepError as e:
        return f"FAILED ({e.step}): {nii}: {e}"


def process_subject(subject, root, output_dir):
    """Run process_nii for every raw volume of one subject."""
    niis = glob(subject + "/*/*.nii.gz")
    results = []
    for nii in niis:
        result = process_nii(nii, root, output_dir)
        if result is not None:
            results.append(result)
    return results


DEBUG = False
max_workers = 4  # CPU-only steps; GPU steps are serialized separately via gpu_lock

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    todo = subjects[:2] if DEBUG else subjects
    future_to_subject = {executor.submit(process_subject, sub, root, output_dir): sub for sub in todo}

    for future in tqdm(as_completed(future_to_subject), total=len(future_to_subject)):
        subject = future_to_subject[future]
        try:
            for result in future.result():
                print(result)
        except Exception as e:
            print(f"Error processing {subject}: {e}")

print("---------  Done ---------")


In [ ]:
!ls ${RAW_DATA_DIR}/Datasets/medical/Brain/OASIS/processed/30863/*

In [ ]:
# Debug: run the pipeline on a single sample file, step by step, so failures are easy to
# pinpoint. Uses the same pipeline.* functions as the real run above (not just printed
# commands), so this actually verifies the setup rather than just echoing stale strings.
sample_subject = subjects[0]
sample_niis = glob(sample_subject + "/*/*.nii.gz")
assert sample_niis, f"no .nii.gz found under {sample_subject}"
sample_nii = sample_niis[0]
print("sample file:", sample_nii)

base = _strip_nii_suffix(sample_nii.replace(root, output_dir))
biasfield = Path(base + "_step1_biasfield.nii.gz")
stripped = Path(base + "_step2_stripped.nii.gz")
registered = Path(base + "_step3_registered.nii.gz")
synthseg = Path(base + "_synthseg.nii.gz")
final = Path(base + "_final.nii.gz")

print("\nRunning N4BiasFieldCorrection...")
pipeline.n4_bias_correction(Path(sample_nii), biasfield)

print("\nRunning mri_synthstrip (GPU)...")
pipeline.skull_strip(biasfield, stripped, gpu=True)

print("\nRunning ANTs registration to", reference_brain)
pipeline.register_to_reference(stripped, registered, Path(reference_brain))

print("\nRunning mri_synthseg (GPU)...")
pipeline.segment(registered, synthseg, gpu=True)

print("\nRunning intensity-normalize whitestripe...")
pipeline.normalize_intensity(registered, final)

print("\nDone:", final)

# Visualize the results

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt

# Pick a sample this notebook actually produced, instead of a hardcoded path from a
# different cohort's directory layout.
nii = glob(subjects[0] + "/*/*.nii.gz")[0]

base = _strip_nii_suffix(nii.replace(root, output_dir))
step1_output = base + "_step1_biasfield.nii.gz"
step2_output = base + "_step2_stripped.nii.gz"
step3_output = base + "_step3_registered.nii.gz"
step4_output = base + "_synthseg.nii.gz"   # matches pipeline.segment's output name (no "step4" in it)
step5_output = base + "_final.nii.gz"


files = [nii, step1_output, step2_output, step3_output, step4_output, step5_output]
images = [nib.load(f).get_fdata() for f in files]

# Choose a slice index to visualize
slice_index = images[0].shape[2] // 2  # Middle slice in the z-dimension
titles = [
    "Original Image (T1W)", 
    "Step 1: Bias Field Corrected", 
    "Step 2: Skull Stripped", 
    "Step 3: Registered", 
    "Step 4: SynthSeg Processed", 
    "Step 5: Final Output"
]


# Plot the slices side-by-side
fig, axes = plt.subplots(1, len(images), figsize=(15, 5))

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img[:, :, slice_index], cmap="gray")
    ax.axis("off")
    ax.set_title(title, fontsize=10)  # Add a title for each subplot

    
plt.tight_layout()
plt.show()


In [ ]:
for sub in subjects:
    subname = sub.split("/")[-1]
    # <subject_id>/<preprocessing>/<date>/<acquisition_id>/<file_name>.nii
    niis = glob(sub + "/*/*/*/*.nii")
    print(subname, ": number of niis:", len(niis))

Bare-CLI equivalent of the pipeline above, for reference (this repo's own code doesn't use
this script -- it calls `code/pipeline.py` directly):

```bash
#!/usr/bin/env bash
set -euo pipefail
T1W_PATH=$1
OUTPUT_DIR=$2
REFERENCE_IMAGE=$3

echo "Running N4BiasFieldCorrection..."
N4BiasFieldCorrection -d 3 -s 4 -i "$T1W_PATH" -o "$OUTPUT_DIR/T1w_BiasField.nii.gz"

echo "Running mri_synthstrip..."
mri_synthstrip -i "$OUTPUT_DIR/T1w_BiasField.nii.gz" -o "$OUTPUT_DIR/T1w_BiasField_Stripped.nii.gz" -g

echo "Running ANTs Registration..."
antsRegistration -d 3 -f "$REFERENCE_IMAGE" -m "$OUTPUT_DIR/T1w_BiasField_Stripped.nii.gz" -o "$OUTPUT_DIR/OutputPrefix"

echo "Running mri_synthseg..."
mri_synthseg --i "$OUTPUT_DIR/T1w_BiasField_Stripped.nii.gz" --o "$OUTPUT_DIR/SynthSeg_Output.nii.gz" --gpu

echo "Running intensity-normalize..."
intensity-normalize whitestripe "$OUTPUT_DIR/T1w_BiasField_Stripped.nii.gz" -o "$OUTPUT_DIR/Normalized_Output.nii.gz" --modality t1

echo "Processing complete. Outputs are located in $OUTPUT_DIR"
```

## Clean

In [ ]:
import os
import glob

# Root directory
root = os.path.join(os.environ["RAW_DATA_DIR"], "Datasets/medical/Brain/OASIS/processed")

# Define suffixes to look for
suffixes = [
    "_step1_biasfield.nii.gz",
    "_step2_stripped.nii.gz",
    "_step3_registered.nii.gz"
]

# Find all .nii.gz files under root
nii_files = glob.glob(os.path.join(root, "**", "*.nii.gz"), recursive=True)

# Filter and delete matching files
for nii in nii_files:
    for suffix in suffixes:
        if nii.endswith(suffix):
            os.remove(nii)
            print(f"Removed: {nii}")
            # break  # No need to check other suffixes once removed

print("Done")


In [ ]:
import os
import glob
from pathlib import Path
from tqdm import tqdm

# Root and output directories
root = os.path.join(os.environ["RAW_DATA_DIR"], "Datasets/medical/Brain/OASIS/processed")

# Target resolutions -- both kept (disk is not the constraint here)
RESOLUTIONS = [1.0, 1.5]

# Find synthseg and final files (native resolution, pre-resample)
step4_files = glob.glob(os.path.join(root, "**", "*_synthseg.nii.gz"), recursive=True)
final_files = glob.glob(os.path.join(root, "**", "*_final.nii.gz"), recursive=True)

# Map final files by base name
final_map = {os.path.basename(f).replace("_final.nii.gz", ""): f for f in final_files}

print("Found:", len(step4_files), "synthseg files,", len(final_files), "final files")

# Process file pairs -- reuses pipeline.resample (same zoom/dtype logic as the ADNI script)
for step4_path in tqdm(step4_files):
    base_name = os.path.basename(step4_path).replace("_synthseg.nii.gz", "")
    if base_name not in final_map:
        continue

    final_path = final_map[base_name]

    for res in RESOLUTIONS:
        output_root = os.path.join(os.environ["RAW_DATA_DIR"], "Datasets/medical/Brain/OASIS", f"OASISv{res:g}")

        for path, is_seg in [(step4_path, True), (final_path, False)]:
            rel_path = os.path.relpath(path, root)
            output_path = os.path.join(output_root, rel_path)
            pipeline.resample(Path(path), Path(output_path), res, is_segmentation=is_seg)

print("Done")
